# SQL Support Bot - Evaluation Framework

This notebook provides a comprehensive evaluation framework for the multi-agent SQL support bot.

## Structure:
1. **Setup & LangSmith Connection** - Configure tracing
2. **Test Tracing** - Verify traces appear in LangSmith
3. **Create Test Dataset** - Define test cases  
4. **Define Evaluators** - How to measure success
5. **Run Baseline Eval** - Test with original prompts
6. **Comparison Functions** - Compare different configs


## Cell 1: Setup & LangSmith Connection


In [45]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Verify API keys are set
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in .env file")
if not os.getenv("LANGCHAIN_API_KEY"):
    raise ValueError("LANGCHAIN_API_KEY not found in .env file")

# LangSmith configuration - enables automatic tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "sql-support-bot-evals"

print("✅ Environment variables loaded")
print(f"   OPENAI_API_KEY: {os.getenv('OPENAI_API_KEY')[:20]}...")
print(f"   LANGCHAIN_API_KEY: {os.getenv('LANGCHAIN_API_KEY')[:20]}...")
print(f"   LangSmith Project: {os.environ['LANGCHAIN_PROJECT']}")

# Test LangSmith connection
from langsmith import Client

client = Client()
print("\n✅ Successfully connected to LangSmith!")
print("   All graph invocations will now be automatically traced.")


✅ Environment variables loaded
   OPENAI_API_KEY: sk-proj-jXK1SnfKIYfP...
   LANGCHAIN_API_KEY: lsv2_pt_afe4ddb0f990...
   LangSmith Project: sql-support-bot-evals

✅ Successfully connected to LangSmith!
   All graph invocations will now be automatically traced.


## Cell 2: Test Tracing

Run a single query to verify tracing is working.


In [46]:
from agent_config import build_graph, CONFIGS
from langchain_core.messages import HumanMessage

# Build graph with baseline config
print("Building graph with baseline config...")
graph = build_graph(CONFIGS["baseline"])
print("✅ Graph built successfully!")

# Run ONE test query - this will be automatically traced to LangSmith
print("\nRunning test query: 'Find songs by U2'")
print("(This will take ~5-10 seconds...)\n")

result = graph.invoke([HumanMessage(content="Find songs by U2")])

# Show results
print("✅ Query completed!")
print(f"   Total messages in result: {len(result)}")
print(f"   Agents involved: {[m.name for m in result if hasattr(m, 'name') and m.name]}")
print(f"\n   Final response preview:")
print(f"   {result[-1].content[:200]}...")

print("\n" + "="*60)
print("👉 NOW CHECK LANGSMITH UI:")
print("   1. Go to https://smith.langchain.com/")
print("   2. Open project: 'sql-support-bot-evals'")
print("   3. You should see a trace for this query!")
print("   4. Click on it to see full execution details")
print("="*60)


Building graph with baseline config...
✅ Graph built successfully!

Running test query: 'Find songs by U2'
(This will take ~5-10 seconds...)

✅ Query completed!
   Total messages in result: 5
   Agents involved: ['general', 'music', 'get_tracks_by_artist', 'music']

   Final response preview:
   Here are some songs by U2:

1. Zoo Station
2. Even Better Than The Real Thing
3. One
4. Until The End Of The World
5. Who's Gonna Ride Your Wild Horses
6. So Cruel
7. The Fly
8. Mysterious Ways
9. Try...

👉 NOW CHECK LANGSMITH UI:
   1. Go to https://smith.langchain.com/
   2. Open project: 'sql-support-bot-evals'
   3. You should see a trace for this query!
   4. Click on it to see full execution details


## Cell 3: Create Test Dataset

Defines 55 test cases and upload to LangSmith


In [47]:
# Import test definitions from tests.py
from tests import all_tests, TEST_SUMMARY
from langsmith import Client

# Initialize LangSmith client
client = Client()

# ============================================================================
# Display Test Summary
# ============================================================================

print(f"✅ Loaded {TEST_SUMMARY['total']} test cases from tests.py\n")

print("Breakdown by Category:")
for category, count in TEST_SUMMARY['by_category'].items():
    print(f"  {category.replace('_', ' ').title():22s}: {count:2d} tests")

print(f"\nDistribution by Agent:")
for agent, count in TEST_SUMMARY['by_agent'].items():
    pct = count / TEST_SUMMARY['total'] * 100
    print(f"  {agent.title():12s}: {count:2d} tests ({pct:.0f}%)")

# ============================================================================
# Upload to LangSmith
# ============================================================================

dataset_name = "sql-support-bot-tests"

print(f"\n{'='*60}")
print(f"Uploading to LangSmith...")
print(f"{'='*60}\n")

# Delete existing dataset if it exists (for clean re-runs)
try:
    existing_datasets = list(client.list_datasets(dataset_name=dataset_name))
    if existing_datasets:
        print(f"⚠️  Dataset '{dataset_name}' already exists. Deleting and recreating...")
        client.delete_dataset(dataset_id=existing_datasets[0].id)
        print(f"✅ Deleted existing dataset")
except Exception as e:
    print(f"No existing dataset to delete (this is fine)")

# Create new dataset
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description=f"Comprehensive test suite: {TEST_SUMMARY['total']} tests across 8 categories"
)
print(f"✅ Created new dataset: '{dataset_name}'")

# Upload all test cases
print(f"\n⏳ Uploading {len(all_tests)} test cases...")

for i, test in enumerate(all_tests, 1):
    client.create_example(
        dataset_id=dataset.id,
        inputs=test["inputs"],
        outputs=test["outputs"],
        metadata={
            "category": test["outputs"]["category"],
            "subcategory": test["outputs"].get("subcategory", ""),
            "test_id": test["outputs"].get("test_id", f"TEST_{i}"),
        }
    )
    if i % 10 == 0:
        print(f"   Uploaded {i}/{len(all_tests)}...")

print(f"\n✅ Successfully uploaded all {len(all_tests)} test cases!")
print(f"\n{'='*60}")
print(f"👉 VIEW IN LANGSMITH:")
print(f"   1. Go to https://smith.langchain.com/")
print(f"   2. Click 'Datasets' tab")
print(f"   3. Find dataset: '{dataset_name}'")
print(f"   4. Browse and filter by category")
print(f"{'='*60}")
print(f"\n💡 To modify tests: Edit tests.py and re-run this cell")


✅ Loaded 55 test cases from tests.py

Breakdown by Category:
  Happy Path            : 12 tests
  Routing               :  8 tests
  Tool Selection        : 10 tests
  Parameter Extraction  :  5 tests
  Data Handling         :  6 tests
  Edge Cases            :  5 tests
  Error Handling        :  4 tests
  Multi Turn            :  5 tests

Distribution by Agent:
  Music       : 30 tests (55%)
  Customer    : 20 tests (36%)
  General     :  5 tests (9%)

Uploading to LangSmith...

⚠️  Dataset 'sql-support-bot-tests' already exists. Deleting and recreating...
✅ Deleted existing dataset
✅ Created new dataset: 'sql-support-bot-tests'

⏳ Uploading 55 test cases...
   Uploaded 10/55...
   Uploaded 20/55...
   Uploaded 30/55...
   Uploaded 40/55...
   Uploaded 50/55...

✅ Successfully uploaded all 55 test cases!

👉 VIEW IN LANGSMITH:
   1. Go to https://smith.langchain.com/
   2. Click 'Datasets' tab
   3. Find dataset: 'sql-support-bot-tests'
   4. Browse and filter by category

💡 To modify 

## Cell 4: Define Evaluators

Helper functions and evaluator functions for measuring agent performance.


In [48]:
# ============================================================================
# HELPER FUNCTIONS - Extract data from traces
# ============================================================================

def get_agents_from_trace(run):
    """Extract all agents that were invoked in this trace."""
    agents = []
    messages = run.get("outputs", {}).get("output", []) if isinstance(run.get("outputs"), dict) else run.get("output", [])
    
    for msg in messages:
        if hasattr(msg, 'name') and msg.name:
            if msg.name in ['general', 'music', 'customer']:
                agents.append(msg.name)
    
    return agents

def get_tool_calls_from_trace(run):
    """Extract all tools that were called in this trace."""
    tools = []
    messages = run.get("outputs", {}).get("output", []) if isinstance(run.get("outputs"), dict) else run.get("output", [])
    
    for msg in messages:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                # Handle both dict and object formats
                if isinstance(tc, dict):
                    tools.append({
                        'name': tc.get('name'),
                        'args': tc.get('args', {})
                    })
                else:
                    tools.append({
                        'name': getattr(tc, 'name', None),
                        'args': getattr(tc, 'args', {})
                    })
    
    return tools

def get_final_response(run):
    """Extract the final response text from the agent."""
    messages = run.get("outputs", {}).get("output", []) if isinstance(run.get("outputs"), dict) else run.get("output", [])
    
    if messages:
        last_msg = messages[-1]
        if hasattr(last_msg, 'content'):
            return last_msg.content
    
    return ""

def get_tool_results_from_trace(run):
    """Extract all tool results (ToolMessage content)."""
    from langchain_core.messages import ToolMessage
    
    results = []
    messages = run.get("outputs", {}).get("output", []) if isinstance(run.get("outputs"), dict) else run.get("output", [])
    
    for msg in messages:
        if isinstance(msg, ToolMessage):
            results.append({
                'tool': msg.name if hasattr(msg, 'name') else 'unknown',
                'content': msg.content
            })
    
    return results

# ============================================================================
# EVALUATOR FUNCTIONS - Compare expected vs actual
# ============================================================================

def check_routing(run, example):
    """
    Evaluator: Check if the correct agent was routed to.
    
    Expected output keys:
    - expected_agent: str (e.g., "music", "customer", "general")
    """
    expected_agent = example.outputs.get("expected_agent")
    if not expected_agent:
        return None  # Skip if no routing expectation
    
    actual_agents = get_agents_from_trace(run)
    
    return {
        "key": "routing_correct",
        "score": 1 if expected_agent in actual_agents else 0,
        "comment": f"Expected: {expected_agent}, Got: {actual_agents}"
    }

def check_tool_selection(run, example):
    """
    Evaluator: Check if the correct tool was called.
    
    Expected output keys:
    - expected_tool: str (e.g., "get_tracks_by_artist", "get_albums_by_artist")
    """
    expected_tool = example.outputs.get("expected_tool")
    if not expected_tool:
        return None  # Skip if no tool expectation
    
    actual_tools = get_tool_calls_from_trace(run)
    tool_names = [t['name'] for t in actual_tools if t['name']]
    
    return {
        "key": "tool_selection_correct",
        "score": 1 if expected_tool in tool_names else 0,
        "comment": f"Expected: {expected_tool}, Got: {tool_names}"
    }

def check_tool_not_called(run, example):
    """
    Evaluator: Check that a specific tool was NOT called (for negative tests).
    
    Expected output keys:
    - should_not_call: str (e.g., "get_customer_info")
    """
    should_not_call = example.outputs.get("should_not_call")
    if not should_not_call:
        return None  # Skip if no negative expectation
    
    actual_tools = get_tool_calls_from_trace(run)
    tool_names = [t['name'] for t in actual_tools if t['name']]
    
    return {
        "key": "correct_tool_avoidance",
        "score": 0 if should_not_call in tool_names else 1,
        "comment": f"Should NOT call: {should_not_call}, Tools called: {tool_names}"
    }

def check_parameter_extraction(run, example):
    """
    Evaluator: Check if tool was called with correct parameters.
    
    Expected output keys:
    - expected_params: dict (e.g., {"artist": "U2"}, {"customer_id": 5})
    """
    expected_params = example.outputs.get("expected_params")
    if not expected_params:
        return None  # Skip if no parameter expectation
    
    actual_tools = get_tool_calls_from_trace(run)
    
    # Check if any tool call has matching parameters
    for tool in actual_tools:
        args = tool.get('args', {})
        # Check if all expected params are present and match
        if all(args.get(k) == v for k, v in expected_params.items()):
            return {
                "key": "parameter_extraction_correct",
                "score": 1,
                "comment": f"Correct params: {expected_params}"
            }
    
    # Collect actual params for debugging
    actual_params = [tool.get('args', {}) for tool in actual_tools]
    
    return {
        "key": "parameter_extraction_correct",
        "score": 0,
        "comment": f"Expected: {expected_params}, Got: {actual_params}"
    }

def check_response_contains(run, example):
    """
    Evaluator: Check if the final response contains expected content.
    
    Expected output keys:
    - must_contain: str or list[str] (keywords/phrases that should appear)
    """
    must_contain = example.outputs.get("must_contain")
    if not must_contain:
        return None  # Skip if no content expectation
    
    final_response = get_final_response(run).lower()
    
    # Handle both single string and list of strings
    if isinstance(must_contain, str):
        must_contain = [must_contain]
    
    # Check if all required phrases are present
    missing = [phrase for phrase in must_contain if phrase.lower() not in final_response]
    
    return {
        "key": "response_contains_expected",
        "score": 1 if len(missing) == 0 else 0,
        "comment": f"Missing: {missing}" if missing else "All required content present"
    }

def check_response_not_contains(run, example):
    """
    Evaluator: Check that the response does NOT contain certain content.
    
    Expected output keys:
    - must_not_contain: str or list[str] (e.g., error messages, SQL keywords)
    """
    must_not_contain = example.outputs.get("must_not_contain")
    if not must_not_contain:
        return None  # Skip if no negative content expectation
    
    final_response = get_final_response(run).lower()
    
    # Handle both single string and list of strings
    if isinstance(must_not_contain, str):
        must_not_contain = [must_not_contain]
    
    # Check if any forbidden phrases are present
    found = [phrase for phrase in must_not_contain if phrase.lower() in final_response]
    
    return {
        "key": "response_avoids_forbidden",
        "score": 1 if len(found) == 0 else 0,
        "comment": f"Found forbidden: {found}" if found else "No forbidden content"
    }

def check_data_handling(run, example):
    """
    Evaluator: Check if agent properly handles existing vs non-existent data.
    
    Expected output keys:
    - data_exists: bool (True = data should be found, False = should handle gracefully)
    """
    data_exists = example.outputs.get("data_exists")
    if data_exists is None:
        return None  # Skip if no data handling expectation
    
    final_response = get_final_response(run).lower()
    tool_results = get_tool_results_from_trace(run)
    
    # Check if tools returned results
    has_results = any(
        result['content'] and result['content'] != '[]' and 'similar' not in result['content'].lower()
        for result in tool_results
    )
    
    if data_exists:
        # Should have found data
        score = 1 if has_results else 0
        comment = "Data found as expected" if has_results else "Expected data not found"
    else:
        # Should gracefully handle missing data
        graceful_phrases = ["couldn't find", "don't have", "not available", "similar"]
        handles_gracefully = any(phrase in final_response for phrase in graceful_phrases)
        score = 1 if handles_gracefully else 0
        comment = "Gracefully handled missing data" if handles_gracefully else "Did not handle missing data gracefully"
    
    return {
        "key": "data_handling_correct",
        "score": score,
        "comment": comment
    }

def check_no_errors(run, example):
    """
    Evaluator: Check that no errors occurred during execution.
    """
    final_response = get_final_response(run).lower()
    
    error_indicators = ["error", "exception", "traceback", "failed"]
    has_errors = any(indicator in final_response for indicator in error_indicators)
    
    return {
        "key": "no_errors",
        "score": 0 if has_errors else 1,
        "comment": "Errors detected" if has_errors else "No errors"
    }

# ============================================================================
# AGGREGATE EVALUATOR - Run all applicable checks
# ============================================================================

def evaluate_run(run, example):
    """
    Run all applicable evaluators on a single test case.
    
    Returns a dict of scores for each metric.
    """
    evaluators = [
        check_routing,
        check_tool_selection,
        check_tool_not_called,
        check_parameter_extraction,
        check_response_contains,
        check_response_not_contains,
        check_data_handling,
        check_no_errors
    ]
    
    results = {}
    
    for evaluator in evaluators:
        result = evaluator(run, example)
        if result:  # Only include if evaluator returned a result
            results[result["key"]] = {
                "score": result["score"],
                "comment": result.get("comment", "")
            }
    
    return results

print("✅ Evaluator functions defined:")
print("   - Helper functions: 4")
print("   - Individual evaluators: 8")
print("   - Aggregate evaluator: 1")
print("\n💡 These will be used to measure agent performance across all 55 tests")


✅ Evaluator functions defined:
   - Helper functions: 4
   - Individual evaluators: 8
   - Aggregate evaluator: 1

💡 These will be used to measure agent performance across all 55 tests


## Cell 5: Run Baseline Evaluation

Run all 55 tests against the baseline configuration and display results.


In [49]:
from agent_config import build_graph, CONFIGS
from langchain_core.messages import HumanMessage
from langsmith import Client
from collections import defaultdict
import time
from tqdm.auto import tqdm

# ============================================================================
# EVALUATION HARNESS
# ============================================================================

def run_evaluation(config_name="baseline", max_tests=None, category_filter=None):
    """
    Run evaluation on the dataset.
    
    Args:
        config_name: Which config to test (e.g., "baseline", "current")
        max_tests: Limit number of tests (None = all tests)
        category_filter: Only run tests from specific category (None = all)
    
    Returns:
        dict with aggregated results
    """
    print(f"\n{'='*70}")
    print(f"🚀 RUNNING EVALUATION: {config_name.upper()}")
    print(f"{'='*70}\n")
    
    # Build graph with specified config
    print(f"Building graph with '{config_name}' config...")
    graph = build_graph(CONFIGS[config_name])
    print("✅ Graph built\n")
    
    # Load dataset from LangSmith
    client = Client()
    dataset_name = "sql-support-bot-tests"
    
    print(f"Loading dataset: '{dataset_name}'...")
    examples = list(client.list_examples(dataset_name=dataset_name))
    print(f"✅ Loaded {len(examples)} test cases\n")
    
    # Apply filters
    if category_filter:
        examples = [ex for ex in examples if ex.outputs.get("category") == category_filter]
        print(f"📋 Filtering to category '{category_filter}': {len(examples)} tests\n")
    
    if max_tests:
        examples = examples[:max_tests]
        print(f"📋 Limiting to first {max_tests} tests\n")
    
    # Run tests
    print(f"{'─'*70}")
    print(f"Running {len(examples)} tests...")
    print(f"{'─'*70}\n")
    
    results = []
    category_scores = defaultdict(lambda: {"passed": 0, "total": 0})
    metric_scores = defaultdict(lambda: {"passed": 0, "total": 0})
    
    start_time = time.time()
    
    for i, example in enumerate(tqdm(examples, desc=f"Evaluating {config_name}", unit="test"), 1):
        test_id = example.outputs.get("test_id", f"TEST_{i}")
        category = example.outputs.get("category", "unknown")
        
        try:
            # Run the agent
            messages = example.inputs["messages"]
            output = graph.invoke(messages)
            
            # Create a "run" object that evaluators can parse
            run = {
                "outputs": {"output": output},
                "example_id": example.id
            }
            
            # Evaluate
            eval_results = evaluate_run(run, example)
            
            # Track results
            test_passed = all(r["score"] == 1 for r in eval_results.values())
            
            results.append({
                "test_id": test_id,
                "category": category,
                "passed": test_passed,
                "metrics": eval_results
            })
            
            # Update category scores
            category_scores[category]["total"] += 1
            if test_passed:
                category_scores[category]["passed"] += 1
            
            # Update metric scores
            for metric, result in eval_results.items():
                metric_scores[metric]["total"] += 1
                if result["score"] == 1:
                    metric_scores[metric]["passed"] += 1
        
        except Exception as e:
            print(f"\n❌ Error on {test_id}: {str(e)}\n")
            results.append({
                "test_id": test_id,
                "category": category,
                "passed": False,
                "error": str(e)
            })
            category_scores[category]["total"] += 1
    
    total_time = time.time() - start_time
    
    # ========================================================================
    # DISPLAY RESULTS
    # ========================================================================
    
    print(f"\n{'='*70}")
    print(f"📊 EVALUATION RESULTS: {config_name.upper()}")
    print(f"{'='*70}\n")
    
    # Overall summary
    total_passed = sum(1 for r in results if r["passed"])
    total_tests = len(results)
    overall_pct = (total_passed / total_tests * 100) if total_tests > 0 else 0
    
    print(f"Overall: {total_passed}/{total_tests} tests passed ({overall_pct:.1f}%)")
    print(f"Time: {total_time:.1f}s ({total_time/total_tests:.2f}s per test)\n")
    
    # Results by category
    print("Results by Category:")
    print(f"{'─'*70}")
    for category in sorted(category_scores.keys()):
        scores = category_scores[category]
        pct = (scores["passed"] / scores["total"] * 100) if scores["total"] > 0 else 0
        status = "✅" if pct == 100 else "⚠️" if pct >= 70 else "❌"
        print(f"{status} {category.replace('_', ' ').title():25s}: {scores['passed']:2d}/{scores['total']:2d} ({pct:5.1f}%)")
    
    # Results by metric
    print(f"\nResults by Metric:")
    print(f"{'─'*70}")
    for metric in sorted(metric_scores.keys()):
        scores = metric_scores[metric]
        pct = (scores["passed"] / scores["total"] * 100) if scores["total"] > 0 else 0
        status = "✅" if pct == 100 else "⚠️" if pct >= 70 else "❌"
        print(f"{status} {metric.replace('_', ' ').title():30s}: {scores['passed']:2d}/{scores['total']:2d} ({pct:5.1f}%)")
    
    # Failed tests detail
    failed_tests = [r for r in results if not r["passed"]]
    if failed_tests:
        print(f"\n{'─'*70}")
        print(f"❌ Failed Tests ({len(failed_tests)}):")
        print(f"{'─'*70}")
        for test in failed_tests[:10]:  # Show first 10 failures
            print(f"\n{test['test_id']} ({test['category']})")
            if "error" in test:
                print(f"  Error: {test['error']}")
            else:
                for metric, result in test.get("metrics", {}).items():
                    if result["score"] == 0:
                        print(f"  ❌ {metric}: {result['comment']}")
        
        if len(failed_tests) > 10:
            print(f"\n  ... and {len(failed_tests) - 10} more failures")
    
    print(f"\n{'='*70}\n")
    
    return {
        "config": config_name,
        "overall": {"passed": total_passed, "total": total_tests, "percentage": overall_pct},
        "by_category": dict(category_scores),
        "by_metric": dict(metric_scores),
        "failed_tests": failed_tests,
        "time": total_time
    }

# ============================================================================
# RUN BASELINE EVALUATION
# ============================================================================

baseline_results = run_evaluation(config_name="baseline")

print("💾 Results saved to 'baseline_results' variable")
print("💡 To test changes: Update agent_config.py, then run run_evaluation('current')")



🚀 RUNNING EVALUATION: BASELINE

Building graph with 'baseline' config...
✅ Graph built

Loading dataset: 'sql-support-bot-tests'...
✅ Loaded 55 test cases

──────────────────────────────────────────────────────────────────────
Running 55 tests...
──────────────────────────────────────────────────────────────────────



Evaluating baseline: 100%|██████████| 55/55 [01:45<00:00,  1.92s/test]


📊 EVALUATION RESULTS: BASELINE

Overall: 53/55 tests passed (96.4%)
Time: 105.7s (1.92s per test)

Results by Category:
──────────────────────────────────────────────────────────────────────
⚠️ Data Handling            :  5/ 6 ( 83.3%)
✅ Edge Cases               :  5/ 5 (100.0%)
✅ Error Handling           :  4/ 4 (100.0%)
✅ Happy Path               : 12/12 (100.0%)
✅ Multi Turn               :  5/ 5 (100.0%)
⚠️ Parameter Extraction     :  4/ 5 ( 80.0%)
✅ Routing                  :  8/ 8 (100.0%)
✅ Tool Selection           : 10/10 (100.0%)

Results by Metric:
──────────────────────────────────────────────────────────────────────
✅ No Errors                     : 55/55 (100.0%)
✅ Parameter Extraction Correct  : 11/11 (100.0%)
⚠️ Routing Correct               : 53/55 ( 96.4%)
✅ Tool Selection Correct        : 21/21 (100.0%)

──────────────────────────────────────────────────────────────────────
❌ Failed Tests (2):
──────────────────────────────────────────────────────────────────────

E3

## Cell 6: View Baseline Results

Simple summary of baseline evaluation performance.


In [50]:
# Overall Performance
print(f"Overall: {baseline_results['overall']['passed']}/{baseline_results['overall']['total']} ({baseline_results['overall']['percentage']:.1f}%)")
print(f"Time: {baseline_results['time']:.1f}s\n")

# By Category (in order A-H)
category_labels = {
    'happy_path': 'A. Happy Path',
    'routing': 'B. Routing',
    'tool_selection': 'C. Tool Selection',
    'parameter_extraction': 'D. Parameter Extraction',
    'data_handling': 'E. Data Handling',
    'edge_cases': 'F. Edge Cases',
    'error_handling': 'G. Error Handling',
    'multi_turn': 'H. Multi-Turn'
}

print("By Category:")
for category, label in category_labels.items():
    if category in baseline_results['by_category']:
        scores = baseline_results['by_category'][category]
        pct = (scores['passed'] / scores['total'] * 100) if scores['total'] > 0 else 0
        print(f"  {label:30s} {scores['passed']:2d}/{scores['total']:2d} ({pct:.1f}%)")

# Failed Tests Summary
failed = baseline_results['failed_tests']
if failed:
    print(f"\nFailed: {len(failed)} tests")
    for test in failed:
        print(f"  • {test['test_id']} ({test['category']})")


Overall: 53/55 (96.4%)
Time: 105.7s

By Category:
  A. Happy Path                  12/12 (100.0%)
  B. Routing                      8/ 8 (100.0%)
  C. Tool Selection              10/10 (100.0%)
  D. Parameter Extraction         4/ 5 (80.0%)
  E. Data Handling                5/ 6 (83.3%)
  F. Edge Cases                   5/ 5 (100.0%)
  G. Error Handling               4/ 4 (100.0%)
  H. Multi-Turn                   5/ 5 (100.0%)

Failed: 2 tests
  • E3.1 (data_handling)
  • D4 (parameter_extraction)


### Cell 6.5 Inspect Failed Tests (Optional)

Shows detailed expected vs actual for each failure.


In [51]:
# Detailed failure inspection
from tests import all_tests

print("Failure Analysis:")
print("=" * 70)

for failure in baseline_results['failed_tests']:
    test_id = failure['test_id']
    
    # Find the test definition
    test_def = next((t for t in all_tests if t['outputs']['test_id'] == test_id), None)
    
    if not test_def:
        print(f"\n❌ {test_id}: Test definition not found")
        continue
    
    print(f"\n❌ {test_id}: {test_def['outputs'].get('description', 'No description')}")
    print(f"   Category: {failure['category']}")
    print(f"   Query: {test_def['inputs']['messages'][0].content}")
    
    # Show expected values
    print(f"\n   Expected:")
    outputs = test_def['outputs']
    for key in ['expected_agent', 'expected_tool', 'expected_params', 'must_contain', 'should_not_call']:
        if key in outputs:
            print(f"     • {key}: {outputs[key]}")
    
    # Show what failed
    print(f"\n   What Failed:")
    if 'error' in failure:
        print(f"     • Error: {failure['error']}")
    else:
        for metric, result in failure.get('metrics', {}).items():
            if result['score'] == 0:
                print(f"     • {metric}: {result['comment']}")
    
    print()

print("=" * 70)
print(f"\n💡 To debug a specific test:")
print(f"   test_single_query('your query here', 'baseline')")


Failure Analysis:

❌ E3.1: Typo should fuzzy match via LIKE
   Category: data_handling
   Query: Songs by Beetles

   Expected:
     • expected_agent: music

   What Failed:
     • routing_correct: Expected: music, Got: ['general']


❌ D4: Customer ID from natural language
   Category: parameter_extraction
   Query: I'm customer number 42

   Expected:
     • expected_agent: customer

   What Failed:
     • routing_correct: Expected: customer, Got: ['general']


💡 To debug a specific test:
   test_single_query('your query here', 'baseline')


## Cell 7: Modular Testing Helpers

Quick functions for testing specific categories or comparing configs.


In [103]:
# ============================================================================
# QUICK TESTING HELPERS
# ============================================================================

def test_category(category, config_name="current"):
    """
    Quickly test a specific category of tests.
    
    Example:
        test_category("tool_selection", "current")
    """
    print(f"\n🔍 Testing category: {category}")
    return run_evaluation(config_name=config_name, category_filter=category)

def test_sample(n=8, config_name="current", strategy="stratified"):
    """
    Quick smoke test - samples tests across all categories.
    
    Args:
        n: Number of tests (default 8 = 1 per category)
        config_name: Which config to test
        strategy: 'stratified' (spread across categories) or 'first' (sequential)
    
    Examples:
        test_sample(8)           # 1 test from each of 8 categories
        test_sample(16)          # 2 tests from each category
        test_sample(5, strategy="first")  # Old behavior: first 5 tests
    """
    if strategy == "first":
        print(f"\n🔍 Testing first {n} tests (sequential)")
        return run_evaluation(config_name=config_name, max_tests=n)
    
    # Stratified sampling: take tests from all categories
    print(f"\n🔍 Stratified sampling: {n} tests across all categories")
    
    # Load all examples
    client = Client()
    dataset_name = "sql-support-bot-tests"
    all_examples = list(client.list_examples(dataset_name=dataset_name))
    
    # Group by category
    categories = ["happy_path", "routing", "tool_selection", "parameter_extraction",
                  "data_handling", "edge_cases", "error_handling", "multi_turn"]
    
    by_category = {cat: [] for cat in categories}
    for ex in all_examples:
        cat = ex.outputs.get("category")
        if cat in by_category:
            by_category[cat].append(ex)
    
    # Take n/8 tests from each category (minimum 1 per category if n >= 8)
    per_category = max(1, n // len(categories))
    
    sampled = []
    for cat in categories:
        sampled.extend(by_category[cat][:per_category])
        if len(sampled) >= n:
            break
    
    sampled = sampled[:n]
    
    # Print what we're testing
    print(f"Sampling breakdown:")
    for cat in categories:
        count = sum(1 for ex in sampled if ex.outputs.get("category") == cat)
        if count > 0:
            print(f"  {cat.replace('_', ' ').title():25s}: {count} test(s)")
    print()
    
    # Build graph
    graph = build_graph(CONFIGS[config_name])
    
    # Run evaluation on sampled tests (simplified version of run_evaluation)
    results = []
    category_scores = defaultdict(lambda: {"passed": 0, "total": 0})
    
    for i, example in enumerate(sampled, 1):
        test_id = example.outputs.get("test_id", f"TEST_{i}")
        category = example.outputs.get("category", "unknown")
        
        try:
            # Run test
            result = graph.invoke(example.inputs["messages"])
            
            # Create run object for evaluation
            run = {"outputs": {"output": result}}
            
            # Evaluate
            eval_results = evaluate_run(run, example)
            
            # Check if passed
            passed = all(r["score"] == 1 for r in eval_results.values() if r is not None)
            
            category_scores[category]["total"] += 1
            if passed:
                category_scores[category]["passed"] += 1
                print(f"✅ {test_id} ({category})")
            else:
                print(f"❌ {test_id} ({category})") 
            
            results.append({"test_id": test_id, "category": category, "passed": passed})
            
        except Exception as e:
            print(f"❌ {test_id} ({category}) - Error: {str(e)[:100]}")
            category_scores[category]["total"] += 1
            results.append({"test_id": test_id, "category": category, "passed": False, "error": str(e)})
    
    # Print summary
    total_passed = sum(1 for r in results if r["passed"])
    total_tests = len(results)
    overall_pct = (total_passed / total_tests * 100) if total_tests > 0 else 0
    
    print(f"\n{'='*70}")
    print(f"📊 SAMPLE RESULTS")
    print(f"{'='*70}")
    print(f"Overall: {total_passed}/{total_tests} ({overall_pct:.1f}%)\n")
    
    print("By Category:")
    for cat in sorted(category_scores.keys()):
        scores = category_scores[cat]
        pct = (scores["passed"] / scores["total"] * 100) if scores["total"] > 0 else 0
        status = "✅" if pct == 100 else "⚠️" if pct >= 70 else "❌"
        print(f"{status} {cat.replace('_', ' ').title():25s}: {scores['passed']}/{scores['total']} ({pct:.1f}%)")
    
    print(f"\n{'='*70}\n")
    
    return {
        "config": config_name,
        "overall": {"passed": total_passed, "total": total_tests, "percentage": overall_pct},
        "by_category": dict(category_scores),
        "results": results
    }

def compare_configs(baseline_config="baseline", new_config="current", category=None):
    """
    Compare two configurations side-by-side.
    
    Example:
        compare_configs("baseline", "current")
        compare_configs("baseline", "current", category="tool_selection")
    """
    print(f"\n{'='*70}")
    print(f"📊 COMPARING CONFIGURATIONS")
    print(f"{'='*70}\n")
    
    # Run both evaluations
    baseline = run_evaluation(baseline_config, category_filter=category)
    new = run_evaluation(new_config, category_filter=category)
    
    # Compare overall
    print(f"\n{'='*70}")
    print(f"🔄 COMPARISON SUMMARY")
    print(f"{'='*70}\n")
    
    print(f"Overall Performance:")
    baseline_pct = baseline["overall"]["percentage"]
    new_pct = new["overall"]["percentage"]
    diff = new_pct - baseline_pct
    
    print(f"  Baseline ({baseline_config}): {baseline['overall']['passed']}/{baseline['overall']['total']} ({baseline_pct:.1f}%)")
    print(f"  New ({new_config}):      {new['overall']['passed']}/{new['overall']['total']} ({new_pct:.1f}%)")
    
    if diff > 0:
        print(f"  📈 Improvement: +{diff:.1f}%")
    elif diff < 0:
        print(f"  📉 Regression: {diff:.1f}%")
    else:
        print(f"  ➡️  No change")
    
    # Compare by category
    print(f"\nBy Category:")
    print(f"{'─'*70}")
    print(f"{'Category':<25} {'Baseline':>12} {'New':>12} {'Change':>10}")
    print(f"{'─'*70}")
    
    all_categories = set(baseline["by_category"].keys()) | set(new["by_category"].keys())
    
    for cat in sorted(all_categories):
        b_scores = baseline["by_category"].get(cat, {"passed": 0, "total": 0})
        n_scores = new["by_category"].get(cat, {"passed": 0, "total": 0})
        
        b_pct = (b_scores["passed"] / b_scores["total"] * 100) if b_scores["total"] > 0 else 0
        n_pct = (n_scores["passed"] / n_scores["total"] * 100) if n_scores["total"] > 0 else 0
        
        diff = n_pct - b_pct
        
        diff_str = f"+{diff:.1f}%" if diff > 0 else f"{diff:.1f}%" if diff < 0 else "  --"
        emoji = "📈" if diff > 0 else "📉" if diff < 0 else "  "
        
        print(f"{cat.replace('_', ' ').title():<25} {b_pct:>11.1f}% {n_pct:>11.1f}% {emoji} {diff_str:>7}")
    
    print(f"\n{'='*70}\n")
    
    return {
        "baseline": baseline,
        "new": new,
        "improvement": diff
    }

def test_single_query(query, config_name="current", show_trace=True):
    """
    Test a single query interactively.
    
    Example:
        test_single_query("Find songs by The Beatles", "current")
    """
    print(f"\n{'='*70}")
    print(f"🧪 TESTING SINGLE QUERY")
    print(f"{'='*70}\n")
    print(f"Config: {config_name}")
    print(f"Query: {query}\n")
    
    # Build graph
    graph = build_graph(CONFIGS[config_name])
    
    # Run query
    result = graph.invoke([HumanMessage(content=query)])
    
    # Extract info
    valid_agents = ['general', 'music', 'customer']
    agents = [m.name for m in result if hasattr(m, 'name') and m.name in valid_agents]
    tools = []
    for msg in result:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                name = tc.get('name') if isinstance(tc, dict) else getattr(tc, 'name', None)
                args = tc.get('args', {}) if isinstance(tc, dict) else getattr(tc, 'args', {})
                tools.append({"name": name, "args": args})
    
    final_response = result[-1].content if hasattr(result[-1], 'content') else "No response"
    
    # Display results
    print(f"Execution Summary:")
    print(f"  Agents invoked: {agents}")
    print(f"  Tools called: {[t['name'] for t in tools]}")
    if tools:
        print(f"  Tool args:")
        for tool in tools:
            if tool['name'] not in ['Router']:  # Skip router
                print(f"    - {tool['name']}: {tool['args']}")
    
    print(f"\nFinal Response:")
    print(f"{'─'*70}")
    print(final_response[:500])
    if len(final_response) > 500:
        print(f"... ({len(final_response)} total chars)")
    print(f"{'─'*70}\n")
    
    if show_trace:
        print(f"Full Message Trace:")
        for i, msg in enumerate(result, 1):
            msg_type = type(msg).__name__
            name = getattr(msg, 'name', 'N/A')
            print(f"  {i}. {msg_type} (name={name})")
    
    print(f"\n{'='*70}\n")
    
    return result

def validate_query(
    query,
    expected_agent=None,
    expected_tool=None,
    expected_params=None,
    must_contain=None,
    must_not_contain=None,
    config="current"
):
    """
    Quick one-off test validation without saving.
    
    Example:
        validate_query(
            query="Find albums by Queen",
            expected_agent="music",
            expected_tool="get_albums_by_artist",
            expected_params={"artist": "Queen"}
        )
    """
    print(f"\n{'='*70}")
    print(f"🧪 VALIDATING QUERY")
    print(f"{'='*70}\n")
    print(f"Query: {query}")
    print(f"Config: {config}\n")
    
    # Build graph
    graph = build_graph(CONFIGS[config])
    
    # Run query
    result = graph.invoke([HumanMessage(content=query)])
    
    # Extract final response
    final_response = result[-1].content if hasattr(result[-1], 'content') else "No response"
    
    # Create a mock example for evaluation
    mock_example = type('obj', (object,), {
        'outputs': {
            'expected_agent': expected_agent,
            'expected_tool': expected_tool,
            'expected_params': expected_params,
            'must_contain': must_contain,
            'must_not_contain': must_not_contain
        }
    })()
    
    # Create run object
    run = {"outputs": {"output": result}}
    
    # Evaluate
    eval_results = evaluate_run(run, mock_example)
    
    # Display results
    print("Expected vs Actual:")
    print(f"{'─'*70}\n")
    
    all_passed = True
    
    for metric, eval_result in eval_results.items():
        if eval_result is not None:
            status = "✅" if eval_result["score"] == 1 else "❌"
            all_passed = all_passed and (eval_result["score"] == 1)
            print(f"{status} {metric.replace('_', ' ').title()}")
            print(f"   {eval_result['comment']}\n")
    
    # Display final response
    print(f"{'─'*70}")
    print("Final Response:")
    print(f"{'─'*70}")
    print(final_response[:500])
    if len(final_response) > 500:
        print(f"... ({len(final_response)} total chars)")
    print(f"{'─'*70}\n")
    
    print(f"{'='*70}")
    if all_passed:
        print("✅ ALL CHECKS PASSED")
    else:
        print("❌ SOME CHECKS FAILED")
    print(f"{'='*70}\n")
    
    return all_passed

# ============================================================================
# USAGE EXAMPLES
# ============================================================================

print("✅ Modular testing helpers loaded!\n")
print("Quick Testing Functions:")
print("  • test_category('tool_selection')     - Test specific category")
print("  • test_sample(8)                       - Stratified: 1 test per category")
print("  • test_sample(16)                      - Stratified: 2 tests per category")
print("  • test_sample(5, strategy='first')     - Sequential: first 5 tests")
print("  • compare_configs('baseline', 'current') - Side-by-side comparison")
print("  • test_single_query('Find U2 songs')   - Interactive single test")
print("  • validate_query('Find albums by Queen', expected_agent='music') - Quick validation")
print("\nExamples:")
print("  results = test_category('tool_selection', 'current')")
print("  results = test_sample(8)  # Tests all 8 categories")
print("  comparison = compare_configs('baseline', 'current')")
print("  test_single_query('What albums does AC/DC have?', 'baseline')")
print("  validate_query('Find albums by Queen', expected_agent='music', expected_tool='get_albums_by_artist')")


✅ Modular testing helpers loaded!

Quick Testing Functions:
  • test_category('tool_selection')     - Test specific category
  • test_sample(8)                       - Stratified: 1 test per category
  • test_sample(16)                      - Stratified: 2 tests per category
  • test_sample(5, strategy='first')     - Sequential: first 5 tests
  • compare_configs('baseline', 'current') - Side-by-side comparison
  • test_single_query('Find U2 songs')   - Interactive single test
  • validate_query('Find albums by Queen', expected_agent='music') - Quick validation

Examples:
  results = test_category('tool_selection', 'current')
  results = test_sample(8)  # Tests all 8 categories
  comparison = compare_configs('baseline', 'current')
  test_single_query('What albums does AC/DC have?', 'baseline')
  validate_query('Find albums by Queen', expected_agent='music', expected_tool='get_albums_by_artist')


## Cell 8: Iteration and Compre_Configs

### Available Functions:  
test_single_query     - single query  
test_sample           - stratified sample  
test_category         - category test  
validate_query        - validate single query  
compare_configs       - baseline vs current  
run_evaluation        - Full evaluation  

### Reload Agent Config (if there is a new change)

In [111]:
import importlib
import agent_config
importlib.reload(agent_config)
from agent_config import build_graph, CONFIGS

print("Reloaded agent_config.py")

Reloaded agent_config.py


In [112]:
test_single_query("Find me songs by Taylor Swift","current")


🧪 TESTING SINGLE QUERY

Config: current
Query: Find me songs by Taylor Swift

Execution Summary:
  Agents invoked: ['general', 'music', 'music']
  Tools called: ['Router', 'get_tracks_by_artist']
  Tool args:
    - get_tracks_by_artist: {'artist': 'Taylor Swift'}

Final Response:
──────────────────────────────────────────────────────────────────────
I couldn't find specific songs by Taylor Swift at the moment. If you have any other requests or need help with something else, feel free to ask!
──────────────────────────────────────────────────────────────────────

Full Message Trace:
  1. HumanMessage (name=None)
  2. AIMessage (name=general)
  3. AIMessage (name=music)
  4. ToolMessage (name=get_tracks_by_artist)
  5. AIMessage (name=music)




[HumanMessage(content='Find me songs by Taylor Swift', additional_kwargs={}, response_metadata={}, id='6032bea9-6490-4686-8798-9b8d3e6e2644'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 200, 'total_tokens': 213, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_cbf1785567', 'id': 'chatcmpl-CfLlX79vs3SO9gwyLYZxkVjoD1sDt', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='general', id='lc_run--3f7e4e2c-c605-4292-b031-6ded46dc9959-0', tool_calls=[{'name': 'Router', 'args': {'choice': 'music'}, 'id': 'call_5Ldo8TookPDda5fAVmVWdbCn', 'type': 'tool_call'}], usage_metadata={'input_tokens': 200, 'output_tokens': 13, 'total_token

In [113]:
test_category("happy_path", "current")


🔍 Testing category: happy_path

🚀 RUNNING EVALUATION: CURRENT

Building graph with 'current' config...
✅ Graph built

Loading dataset: 'sql-support-bot-tests'...
✅ Loaded 55 test cases

📋 Filtering to category 'happy_path': 12 tests

──────────────────────────────────────────────────────────────────────
Running 12 tests...
──────────────────────────────────────────────────────────────────────



Evaluating current: 100%|██████████| 12/12 [00:41<00:00,  3.44s/test]


📊 EVALUATION RESULTS: CURRENT

Overall: 12/12 tests passed (100.0%)
Time: 41.3s (3.44s per test)

Results by Category:
──────────────────────────────────────────────────────────────────────
✅ Happy Path               : 12/12 (100.0%)

Results by Metric:
──────────────────────────────────────────────────────────────────────
✅ No Errors                     : 12/12 (100.0%)
✅ Parameter Extraction Correct  :  7/ 7 (100.0%)
✅ Routing Correct               : 12/12 (100.0%)
✅ Tool Selection Correct        : 10/10 (100.0%)




{'config': 'current',
 'overall': {'passed': 12, 'total': 12, 'percentage': 100.0},
 'by_category': {'happy_path': {'passed': 12, 'total': 12}},
 'by_metric': {'routing_correct': {'passed': 12, 'total': 12},
  'no_errors': {'passed': 12, 'total': 12},
  'tool_selection_correct': {'passed': 10, 'total': 10},
  'parameter_extraction_correct': {'passed': 7, 'total': 7}},
 'failed_tests': [],
 'time': 41.29687285423279}

In [114]:
compare_configs("baseline", "current")


📊 COMPARING CONFIGURATIONS


🚀 RUNNING EVALUATION: BASELINE

Building graph with 'baseline' config...
✅ Graph built

Loading dataset: 'sql-support-bot-tests'...
✅ Loaded 55 test cases

──────────────────────────────────────────────────────────────────────
Running 55 tests...
──────────────────────────────────────────────────────────────────────



Evaluating baseline: 100%|██████████| 55/55 [02:12<00:00,  2.41s/test]



📊 EVALUATION RESULTS: BASELINE

Overall: 53/55 tests passed (96.4%)
Time: 132.6s (2.41s per test)

Results by Category:
──────────────────────────────────────────────────────────────────────
⚠️ Data Handling            :  5/ 6 ( 83.3%)
✅ Edge Cases               :  5/ 5 (100.0%)
✅ Error Handling           :  4/ 4 (100.0%)
✅ Happy Path               : 12/12 (100.0%)
✅ Multi Turn               :  5/ 5 (100.0%)
⚠️ Parameter Extraction     :  4/ 5 ( 80.0%)
✅ Routing                  :  8/ 8 (100.0%)
✅ Tool Selection           : 10/10 (100.0%)

Results by Metric:
──────────────────────────────────────────────────────────────────────
✅ No Errors                     : 55/55 (100.0%)
✅ Parameter Extraction Correct  : 11/11 (100.0%)
⚠️ Routing Correct               : 53/55 ( 96.4%)
✅ Tool Selection Correct        : 21/21 (100.0%)

──────────────────────────────────────────────────────────────────────
❌ Failed Tests (2):
──────────────────────────────────────────────────────────────────────

E3

Evaluating current: 100%|██████████| 55/55 [02:22<00:00,  2.59s/test]


📊 EVALUATION RESULTS: CURRENT

Overall: 54/55 tests passed (98.2%)
Time: 142.5s (2.59s per test)

Results by Category:
──────────────────────────────────────────────────────────────────────
✅ Data Handling            :  6/ 6 (100.0%)
✅ Edge Cases               :  5/ 5 (100.0%)
✅ Error Handling           :  4/ 4 (100.0%)
✅ Happy Path               : 12/12 (100.0%)
✅ Multi Turn               :  5/ 5 (100.0%)
⚠️ Parameter Extraction     :  4/ 5 ( 80.0%)
✅ Routing                  :  8/ 8 (100.0%)
✅ Tool Selection           : 10/10 (100.0%)

Results by Metric:
──────────────────────────────────────────────────────────────────────
✅ No Errors                     : 55/55 (100.0%)
✅ Parameter Extraction Correct  : 11/11 (100.0%)
⚠️ Routing Correct               : 54/55 ( 98.2%)
✅ Tool Selection Correct        : 21/21 (100.0%)

──────────────────────────────────────────────────────────────────────
❌ Failed Tests (1):
──────────────────────────────────────────────────────────────────────

D4 (

{'baseline': {'config': 'baseline',
  'overall': {'passed': 53, 'total': 55, 'percentage': 96.36363636363636},
  'by_category': {'multi_turn': {'passed': 5, 'total': 5},
   'error_handling': {'passed': 4, 'total': 4},
   'edge_cases': {'passed': 5, 'total': 5},
   'data_handling': {'passed': 5, 'total': 6},
   'parameter_extraction': {'passed': 4, 'total': 5},
   'tool_selection': {'passed': 10, 'total': 10},
   'routing': {'passed': 8, 'total': 8},
   'happy_path': {'passed': 12, 'total': 12}},
  'by_metric': {'routing_correct': {'passed': 53, 'total': 55},
   'no_errors': {'passed': 55, 'total': 55},
   'tool_selection_correct': {'passed': 21, 'total': 21},
   'parameter_extraction_correct': {'passed': 11, 'total': 11}},
  'failed_tests': [{'test_id': 'E3.1',
    'category': 'data_handling',
    'passed': False,
    'metrics': {'routing_correct': {'score': 0,
      'comment': "Expected: music, Got: ['general']"},
     'no_errors': {'score': 1, 'comment': 'No errors'}}},
   {'test_id'

In [ ]:
compare_configs("baseline", "current", "tool_selection")

In [ ]:
validate_query(
    query="I'm customer 5. How many invoices do I have and what's my country?",
    expected_agent="customer",
    expected_tool="get_customer_invoice_summary",
    expected_params={"customer_id": 5},
    config="current",
    must_contain=["country", "invoice"],
    must_not_contain=["error", "cannot help"]
)


🧪 VALIDATING QUERY

Query: I'm customer 5. How many invoices do I have and what's my country?
Config: baseline

Expected vs Actual:
──────────────────────────────────────────────────────────────────────

✅ Routing Correct
   Expected: customer, Got: ['general', 'customer', 'customer']

❌ Tool Selection Correct
   Expected: get_customer_invoice_summary, Got: ['Router', 'get_customer_info']

✅ Parameter Extraction Correct
   Correct params: {'customer_id': 5}

✅ Response Contains Expected
   All required content present

✅ Response Avoids Forbidden
   No forbidden content

✅ No Errors
   No errors

──────────────────────────────────────────────────────────────────────
Final Response:
──────────────────────────────────────────────────────────────────────
You have 4 invoices, and your country is the Czech Republic.
──────────────────────────────────────────────────────────────────────

❌ SOME CHECKS FAILED



False

In [117]:
test_single_query("I'm customer 8, what's my address? Also, can you list albums by taylor swift?", "current")


🧪 TESTING SINGLE QUERY

Config: current
Query: I'm customer 8, what's my address? Also, can you list albums by taylor swift?



ValueError: General agent should only call one tool